# 1. Comprensión del problema

**Objetivo:** Predecir si un paciente con COVID-19 confirmado en CDMX requerirá
hospitalización o podrá tratarse de forma ambulatoria, usando variables
demográficas y clínicas disponibles al momento de la atención.

**Variable objetivo:** `hospitalizado` (derivada de `TIPO_PACIENTE`)
- `0` → Ambulatorio (`TIPO_PACIENTE = 1`)
- `1` → Hospitalizado (`TIPO_PACIENTE = 2`)

**Diseño experimental (ablación):** se entrenarán y compararán dos modelos:
- **Modelo A — Triaje realista:** incluye `NEUMONIA` (refleja la información
  real disponible en el triaje).
- **Modelo B — Perfil de base:** excluye `NEUMONIA` (mide cuánto se puede
  anticipar el riesgo solo con edad, sexo y comorbilidades crónicas).
La comparación de métricas entre A y B es un resultado del estudio, no un
detalle técnico.

# 2. Selección de datos

**Universo:** solo casos COVID confirmados (`CLASIFICACION_FINAL_COVID` ∈
positivos). Archivo ya filtrado a CDMX; `MUNICIPIO` = 16 alcaldías.

**Variables predictoras (X):**
EDAD, SEXO, EMBARAZO, DIAS_SINTOMAS, DIABETES, HIPERTENSION, OBESIDAD, EPOC,
ASMA, CARDIOVASCULAR, RENAL_CRONICA, INMUSUPR, TABAQUISMO, OTRA_COM.
- `MUNICIPIO`: candidata (decidir si predictora o solo descriptiva).
- `NEUMONIA`: predictora **condicional** — presente solo en el Modelo A
  (objeto del experimento de ablación; justificación documentada en paso 5).

**Variable objetivo (y):** `hospitalizado`

**Excluidas por fuga de información** (ocurren después de decidir hospitalizar):
UCI, INTUBADO, FECHA_DEF, DIAS_ATENCION.
**Excluidas por constancia** (filtro CDMX): ENTIDAD_NAC, ENTIDAD_RES, ENTIDAD_UM.

In [2]:
import polars as pl 
from pathlib import Path
path = Path().cwd().parent / "data" / "polars" / "mexico_city_covid_data_dates.parquet"
print(path)
df = pl.read_parquet(path)

print(df.shape)

/home/netfoor/Documents/university/KDD/covid19/data/polars/mexico_city_covid_data_dates.parquet
(7840211, 45)


In [12]:
print(df["CLASIFICACION_FINAL_COVID"].value_counts(sort=True))
print("")
print(df["TIPO_PACIENTE"].value_counts(sort=True))


shape: (7, 2)
┌───────────────────────────┬─────────┐
│ CLASIFICACION_FINAL_COVID ┆ count   │
│ ---                       ┆ ---     │
│ str                       ┆ u32     │
╞═══════════════════════════╪═════════╡
│ 7                         ┆ 5185103 │
│ 3                         ┆ 2135769 │
│ 1                         ┆ 256054  │
│ 6                         ┆ 222924  │
│ 5                         ┆ 27296   │
│ 4                         ┆ 8176    │
│ 2                         ┆ 4889    │
└───────────────────────────┴─────────┘

shape: (2, 2)
┌───────────────┬─────────┐
│ TIPO_PACIENTE ┆ count   │
│ ---           ┆ ---     │
│ str           ┆ u32     │
╞═══════════════╪═════════╡
│ 1             ┆ 7494753 │
│ 2             ┆ 345458  │
└───────────────┴─────────┘


In [13]:
confirmados = df.filter(pl.col("CLASIFICACION_FINAL_COVID").is_in(["1", "2", "3"]))
print(f"Confirmados: {confirmados.height:,} de {df.height:,}")

Confirmados: 2,396,712 de 7,840,211


In [14]:
balance = (
    confirmados["TIPO_PACIENTE"]
    .value_counts(sort=True)
    .with_columns((pl.col("count") / pl.col("count").sum() * 100).round(2).alias("pct"))
)
print(balance)

shape: (2, 3)
┌───────────────┬─────────┬───────┐
│ TIPO_PACIENTE ┆ count   ┆ pct   │
│ ---           ┆ ---     ┆ ---   │
│ str           ┆ u32     ┆ f64   │
╞═══════════════╪═════════╪═══════╡
│ 1             ┆ 2233295 ┆ 93.18 │
│ 2             ┆ 163417  ┆ 6.82  │
└───────────────┴─────────┴───────┘


In [11]:
print(sorted(df.collect_schema().names()))

['ASMA', 'CARDIOVASCULAR', 'CLASIFICACION_FINAL_COVID', 'CLASIFICACION_FINAL_FLU', 'DIABETES', 'DIAS_ATENCION', 'DIAS_ENFERMO', 'DIAS_SINTOMAS', 'EDAD', 'EMBARAZO', 'ENTIDAD_NAC', 'ENTIDAD_RES', 'ENTIDAD_UM', 'EPOC', 'FECHA_DEF', 'FECHA_INGRESO', 'FECHA_SINTOMAS', 'HABLA_LENGUA_INDIG', 'HIPERTENSION', 'INDIGENA', 'INMUSUPR', 'INTUBADO', 'MIGRANTE', 'MUNICIPIO_RES', 'NACIONALIDAD', 'NEUMONIA', 'OBESIDAD', 'ORIGEN', 'OTRA_COM', 'OTRO_CASO', 'PAIS_NACIONALIDAD', 'PAIS_ORIGEN', 'RENAL_CRONICA', 'RESULTADO_ANTIGENO', 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION', 'SECTOR', 'SEXO', 'TABAQUISMO', 'TIPO_PACIENTE', 'TOMA_MUESTRA_ANTIGENO', 'TOMA_MUESTRA_LAB', 'UCI', 'YEAR_ORIGIN', '_id']


In [ ]:

predictoras = [
    "EDAD", "SEXO", "EMBARAZO", "DIAS_SINTOMAS", "NEUMONIA",
    "DIABETES", "HIPERTENSION", "OBESIDAD", "EPOC", "ASMA",
    "CARDIOVASCULAR", "RENAL_CRONICA", "INMUSUPR", "TABAQUISMO",
    "OTRA_COM", "MUNICIPIO_RES",
]
resumen = []
for c in predictoras:
    s = confirmados[c]
    cod = s.is_in([97, 98, 99]).sum() if s.dtype.is_numeric() else None
    resumen.append({
        "columna": c, "dtype": str(s.dtype),
        "nulos": s.null_count(), "cod_97_98_99": cod,
    })

df_resumen = pl.DataFrame(resumen)

with pl.Config(tbl_rows=-1):
    print(df_resumen)

shape: (16, 4)
┌────────────────┬────────┬───────┬──────────────┐
│ columna        ┆ dtype  ┆ nulos ┆ cod_97_98_99 │
│ ---            ┆ ---    ┆ ---   ┆ ---          │
│ str            ┆ str    ┆ i64   ┆ i64          │
╞════════════════╪════════╪═══════╪══════════════╡
│ EDAD           ┆ Int64  ┆ 0     ┆ 549          │
│ SEXO           ┆ String ┆ 0     ┆ null         │
│ EMBARAZO       ┆ String ┆ 0     ┆ null         │
│ DIAS_SINTOMAS  ┆ Int64  ┆ 0     ┆ 8            │
│ NEUMONIA       ┆ String ┆ 0     ┆ null         │
│ DIABETES       ┆ String ┆ 0     ┆ null         │
│ HIPERTENSION   ┆ String ┆ 0     ┆ null         │
│ OBESIDAD       ┆ String ┆ 0     ┆ null         │
│ EPOC           ┆ String ┆ 0     ┆ null         │
│ ASMA           ┆ String ┆ 0     ┆ null         │
│ CARDIOVASCULAR ┆ String ┆ 0     ┆ null         │
│ RENAL_CRONICA  ┆ String ┆ 0     ┆ null         │
│ INMUSUPR       ┆ String ┆ 0     ┆ null         │
│ TABAQUISMO     ┆ String ┆ 0     ┆ null         │
│ OTRA_COM      

In [4]:
print(df.schema)

Schema({'_id': Binary, 'ASMA': String, 'CARDIOVASCULAR': String, 'CLASIFICACION_FINAL_COVID': String, 'CLASIFICACION_FINAL_FLU': String, 'DIABETES': String, 'EDAD': Int64, 'EMBARAZO': String, 'ENTIDAD_NAC': String, 'ENTIDAD_RES': String, 'ENTIDAD_UM': String, 'EPOC': String, 'FECHA_DEF': Date, 'FECHA_INGRESO': Date, 'FECHA_SINTOMAS': Date, 'HABLA_LENGUA_INDIG': String, 'HIPERTENSION': String, 'INDIGENA': String, 'INMUSUPR': String, 'INTUBADO': String, 'MIGRANTE': String, 'MUNICIPIO_RES': String, 'NACIONALIDAD': String, 'NEUMONIA': String, 'OBESIDAD': String, 'ORIGEN': String, 'OTRA_COM': String, 'OTRO_CASO': String, 'PAIS_NACIONALIDAD': String, 'PAIS_ORIGEN': String, 'RENAL_CRONICA': String, 'RESULTADO_ANTIGENO': String, 'RESULTADO_PCR': String, 'RESULTADO_PCR_COINFECCION': String, 'SECTOR': String, 'SEXO': String, 'TABAQUISMO': String, 'TIPO_PACIENTE': String, 'TOMA_MUESTRA_ANTIGENO': String, 'TOMA_MUESTRA_LAB': String, 'UCI': String, 'YEAR_ORIGIN': Int64, 'DIAS_SINTOMAS': Int64, 'DIA

In [5]:
with pl.Config(
    tbl_rows=-1,
    tbl_cols=-1,
    fmt_str_lengths=100
):
    print(df.head(5))

shape: (5, 45)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ _id ┆ ASM ┆ CAR ┆ CLA ┆ CLA ┆ DIA ┆ EDA ┆ EMB ┆ ENT ┆ ENT ┆ ENT ┆ EPO ┆ FEC ┆ FEC ┆ FEC ┆ HAB ┆ HIP ┆ IND ┆ INM ┆ INT ┆ MIG ┆ MUN ┆ NAC ┆ NEU ┆ OBE ┆ ORI ┆ OTR ┆ OTR ┆ PAI ┆ PAI ┆ REN ┆ RES ┆ RES ┆ RES ┆ SEC ┆ SEX ┆ TAB ┆ TIP ┆ TOM ┆ TOM ┆ UCI ┆ YEA ┆ DIA ┆ DIA ┆ DIA │
│ --- ┆ A   ┆ DIO ┆ SIF ┆ SIF ┆ BET ┆ D   ┆ ARA ┆ IDA ┆ IDA ┆ IDA ┆ C   ┆ HA_ ┆ HA_ ┆ HA_ ┆ LA_ ┆ ERT ┆ IGE ┆ USU ┆ UBA ┆ RAN ┆ ICI ┆ ION ┆ MON ┆ SID ┆ GEN ┆ A_C ┆ O_C ┆ S_N ┆ S_O ┆ AL_ ┆ ULT ┆ ULT ┆ ULT ┆ TOR ┆ O   ┆ AQU ┆ O_P ┆ A_M ┆ A_M ┆ --- ┆ R_O ┆ S_S ┆ S_A ┆ S_E │
│ bin ┆ --- ┆ VAS ┆ ICA ┆ ICA ┆ ES  ┆ --- ┆ ZO  ┆ D_N ┆ D_R ┆ D_U ┆ --- ┆ DEF ┆ ING ┆ SIN ┆ LEN ┆ ENS ┆ NA  ┆ PR  ┆ DO  ┆ TE  ┆ PIO ┆ ALI ┆ IA  ┆ AD  ┆ --- ┆ OM  ┆ ASO ┆

In [17]:
# --- Valores reales de las predictoras categóricas (String) ---
categoricas = [
    "SEXO", "EMBARAZO", "NEUMONIA", "DIABETES", "HIPERTENSION",
    "OBESIDAD", "EPOC", "ASMA", "CARDIOVASCULAR", "RENAL_CRONICA",
    "INMUSUPR", "TABAQUISMO", "OTRA_COM",
]
for c in categoricas:
    print(f"--- {c} ---")
    print(confirmados[c].value_counts(sort=True))
    print()

--- SEXO ---
shape: (2, 2)
┌──────┬─────────┐
│ SEXO ┆ count   │
│ ---  ┆ ---     │
│ str  ┆ u32     │
╞══════╪═════════╡
│ 1    ┆ 1295328 │
│ 2    ┆ 1101384 │
└──────┴─────────┘

--- EMBARAZO ---
shape: (4, 2)
┌──────────┬─────────┐
│ EMBARAZO ┆ count   │
│ ---      ┆ ---     │
│ str      ┆ u32     │
╞══════════╪═════════╡
│ 2        ┆ 1267475 │
│ 97       ┆ 1101384 │
│ 98       ┆ 14082   │
│ 1        ┆ 13771   │
└──────────┴─────────┘

--- NEUMONIA ---
shape: (3, 2)
┌──────────┬─────────┐
│ NEUMONIA ┆ count   │
│ ---      ┆ ---     │
│ str      ┆ u32     │
╞══════════╪═════════╡
│ 2        ┆ 2262088 │
│ 1        ┆ 128461  │
│ 99       ┆ 6163    │
└──────────┴─────────┘

--- DIABETES ---
shape: (3, 2)
┌──────────┬─────────┐
│ DIABETES ┆ count   │
│ ---      ┆ ---     │
│ str      ┆ u32     │
╞══════════╪═════════╡
│ 2        ┆ 2196184 │
│ 1        ┆ 188091  │
│ 98       ┆ 12437   │
└──────────┴─────────┘

--- HIPERTENSION ---
shape: (3, 2)
┌──────────────┬─────────┐
│ HIPERTENSION ┆ c

In [ ]:
# --- Celda: casos frontera en las numéricas (EDAD y DIAS_SINTOMAS) ---
print("EDAD en 97/98/99:")
print(confirmados.filter(pl.col("EDAD").is_in([97, 98, 99]))["EDAD"].value_counts(sort=True))
print()
print("EDAD - estadística general:")
print(confirmados["EDAD"].describe())
print()
print("DIAS_SINTOMAS en 97/98/99:")
print(
    confirmados
    .filter(pl.col("DIAS_SINTOMAS").is_in([97, 98, 99]))
    .select(["DIAS_SINTOMAS", "EDAD", "TIPO_PACIENTE", "FECHA_SINTOMAS", "FECHA_INGRESO"])
)
print()
print("DIAS_SINTOMAS - estadística general:")
print(confirmados["DIAS_SINTOMAS"].describe())

EDAD en 97/98/99 (sospecho que son edades reales, no códigos):
shape: (3, 2)
┌──────┬───────┐
│ EDAD ┆ count │
│ ---  ┆ ---   │
│ i64  ┆ u32   │
╞══════╪═══════╡
│ 97   ┆ 228   │
│ 98   ┆ 178   │
│ 99   ┆ 143   │
└──────┴───────┘

EDAD - estadística general:
shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ value      │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 2.396712e6 │
│ null_count ┆ 0.0        │
│ mean       ┆ 40.290122  │
│ std        ┆ 17.287274  │
│ min        ┆ 0.0        │
│ 25%        ┆ 27.0       │
│ 50%        ┆ 39.0       │
│ 75%        ┆ 52.0       │
│ max        ┆ 164.0      │
└────────────┴────────────┘

DIAS_SINTOMAS en 97/98/99 (aquí sí sospecho de outliers/errores):
shape: (8, 5)
┌───────────────┬──────┬───────────────┬────────────────┬───────────────┐
│ DIAS_SINTOMAS ┆ EDAD ┆ TIPO_PACIENTE ┆ FECHA_SINTOMAS ┆ FECHA_INGRESO │
│ ---           ┆ ---  ┆ ---           ┆ ---            ┆ ---           │
│ i6

In [22]:
# --- Celda: MUNICIPIO_RES ---
print(f"Valores únicos: {confirmados['MUNICIPIO_RES'].n_unique()}")

df_municipios = confirmados["MUNICIPIO_RES"].value_counts(sort=True)
with pl.Config(tbl_rows=-1):
    print(df_municipios)



Valores únicos: 352
shape: (352, 2)
┌───────────────┬────────┐
│ MUNICIPIO_RES ┆ count  │
│ ---           ┆ ---    │
│ str           ┆ u32    │
╞═══════════════╪════════╡
│ 007           ┆ 283318 │
│ 010           ┆ 246398 │
│ 005           ┆ 245172 │
│ 012           ┆ 193129 │
│ 003           ┆ 121669 │
│ 002           ┆ 112689 │
│ 013           ┆ 111338 │
│ 015           ┆ 108399 │
│ 014           ┆ 96619  │
│ 011           ┆ 95667  │
│ 017           ┆ 89095  │
│ 006           ┆ 86094  │
│ 016           ┆ 70737  │
│ 008           ┆ 59982  │
│ 058           ┆ 59906  │
│ 033           ┆ 57833  │
│ 009           ┆ 42309  │
│ 004           ┆ 39089  │
│ 104           ┆ 38795  │
│ 057           ┆ 28053  │
│ 025           ┆ 17270  │
│ 039           ┆ 16618  │
│ 031           ┆ 13224  │
│ 081           ┆ 12215  │
│ 020           ┆ 11426  │
│ 109           ┆ 9598   │
│ 037           ┆ 8962   │
│ 121           ┆ 8912   │
│ 070           ┆ 7820   │
│ 122           ┆ 7056   │
│ 024           ┆ 6

In [ ]:
# --- Celda: ¿se traslapan los "se ignora" entre comorbilidades? ---
comorbilidades = [
    "DIABETES", "HIPERTENSION", "OBESIDAD", "EPOC", "ASMA",
    "CARDIOVASCULAR", "RENAL_CRONICA", "INMUSUPR", "TABAQUISMO", "OTRA_COM",
]
mask_comorb = pl.any_horizontal([pl.col(c) == "98" for c in comorbilidades])
mask_neumonia = pl.col("NEUMONIA") == "99"
mask_embarazo98 = pl.col("EMBARAZO") == "98"

print("Filas con >=1 '98' en comorbilidades:", confirmados.filter(mask_comorb).height)
print("Filas con NEUMONIA = 99:", confirmados.filter(mask_neumonia).height)
print("Filas con EMBARAZO = 98:", confirmados.filter(mask_embarazo98).height)
print("Filas afectadas por CUALQUIERA de los tres:",
      confirmados.filter(mask_comorb | mask_neumonia | mask_embarazo98).height)

Filas con >=1 '98' en comorbilidades: 21500
Filas con NEUMONIA = 99: 6163
Filas con EMBARAZO = 98: 14082
Filas afectadas por CUALQUIERA de los tres: 32703


In [26]:
# --- Celda: aclarar la definición real de "CDMX" en el filtro ---
print("ENTIDAD_RES:")
df_entidad_res = confirmados["ENTIDAD_RES"].value_counts(sort=True)
with pl.Config(tbl_rows=-1):
    print(df_entidad_res)
print()
print("ENTIDAD_UM:")
df_entidad_um = confirmados["ENTIDAD_UM"].value_counts(sort=True)
with pl.Config(tbl_rows=-1):
    print(df_entidad_um)

ENTIDAD_RES:
shape: (32, 2)
┌─────────────┬─────────┐
│ ENTIDAD_RES ┆ count   │
│ ---         ┆ ---     │
│ str         ┆ u32     │
╞═════════════╪═════════╡
│ 09          ┆ 1926528 │
│ 15          ┆ 351022  │
│ 22          ┆ 22085   │
│ 13          ┆ 17425   │
│ 23          ┆ 8387    │
│ 17          ┆ 6984    │
│ 11          ┆ 6904    │
│ 21          ┆ 5777    │
│ 02          ┆ 5774    │
│ 03          ┆ 5206    │
│ 14          ┆ 4304    │
│ 30          ┆ 3566    │
│ 19          ┆ 3544    │
│ 24          ┆ 3375    │
│ 16          ┆ 2259    │
│ 29          ┆ 2220    │
│ 12          ┆ 2200    │
│ 01          ┆ 1935    │
│ 06          ┆ 1803    │
│ 20          ┆ 1756    │
│ 25          ┆ 1682    │
│ 27          ┆ 1645    │
│ 31          ┆ 1502    │
│ 08          ┆ 1476    │
│ 28          ┆ 1450    │
│ 18          ┆ 1107    │
│ 26          ┆ 996     │
│ 05          ┆ 915     │
│ 07          ┆ 822     │
│ 32          ┆ 812     │
│ 04          ┆ 726     │
│ 10          ┆ 525     │
└─────────

In [27]:
# --- Celda: percentiles para definir topes de outliers ---
print("EDAD:")
for q in [0.99, 0.995, 0.999, 1.0]:
    print(f"  p{q}: {confirmados['EDAD'].quantile(q)}")
print(f"  Pacientes con EDAD > 110: {confirmados.filter(pl.col('EDAD') > 110).height}")
print()
print("DIAS_SINTOMAS:")
for q in [0.90, 0.95, 0.99, 0.995, 1.0]:
    print(f"  p{q}: {confirmados['DIAS_SINTOMAS'].quantile(q)}")
print(f"  Pacientes con DIAS_SINTOMAS > 30: {confirmados.filter(pl.col('DIAS_SINTOMAS') > 30).height}")

EDAD:
  p0.99: 82.0
  p0.995: 86.0
  p0.999: 94.0
  p1.0: 164.0
  Pacientes con EDAD > 110: 297

DIAS_SINTOMAS:
  p0.9: 7.0
  p0.95: 8.0
  p0.99: 14.0
  p0.995: 14.0
  p1.0: 371.0
  Pacientes con DIAS_SINTOMAS > 30: 761


In [28]:
# --- Celda: impacto del filtro CDMX + balance dentro del universo correcto ---
cdmx_atencion = confirmados.filter(pl.col("ENTIDAD_UM") == "09")
cdmx_residencia = confirmados.filter(pl.col("ENTIDAD_RES") == "09")
cdmx_union = confirmados.filter(
    (pl.col("ENTIDAD_UM") == "09") | (pl.col("ENTIDAD_RES") == "09")
)

print(f"Atendidos en CDMX (Opción A): {cdmx_atencion.height:,}")
print(f"Residentes de CDMX (Opción B): {cdmx_residencia.height:,}")
print(f"Unión (Opción C): {cdmx_union.height:,}")
print()

balance_cdmx = (
    cdmx_atencion["TIPO_PACIENTE"]
    .value_counts(sort=True)
    .with_columns((pl.col("count") / pl.col("count").sum() * 100).round(2).alias("pct"))
)
print("Balance hospitalizado/ambulatorio dentro de 'atendidos en CDMX':")
print(balance_cdmx)

Atendidos en CDMX (Opción A): 2,165,870
Residentes de CDMX (Opción B): 1,926,528
Unión (Opción C): 2,185,118

Balance hospitalizado/ambulatorio dentro de 'atendidos en CDMX':
shape: (2, 3)
┌───────────────┬─────────┬───────┐
│ TIPO_PACIENTE ┆ count   ┆ pct   │
│ ---           ┆ ---     ┆ ---   │
│ str           ┆ u32     ┆ f64   │
╞═══════════════╪═════════╪═══════╡
│ 1             ┆ 2028831 ┆ 93.67 │
│ 2             ┆ 137039  ┆ 6.33  │
└───────────────┴─────────┴───────┘


# 3. limpieza de datos

In [3]:
print(pl.scan_parquet(path).collect_schema().len(), "columnas en el archivo crudo")
print(pl.scan_parquet(path).select(pl.len()).collect().item(), "filas en el archivo crudo")

45 columnas en el archivo crudo
7840211 filas en el archivo crudo


In [4]:
import polars as pl
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(300)
pl.Config.set_fmt_str_lengths(50)
from pathlib import Path
 
path = Path().cwd().parent / "data" / "polars" / "mexico_city_covid_data_dates.parquet"
 
comorbilidades = [
    "DIABETES", "HIPERTENSION", "OBESIDAD", "EPOC", "ASMA",
    "CARDIOVASCULAR", "RENAL_CRONICA", "INMUSUPR", "TABAQUISMO", "OTRA_COM",
]
 
lf_raw = pl.scan_parquet(path)
 
# --- Filtros compartidos, definidos una vez, reutilizados por ambos ---
filtro_confirmados_cdmx = (
    pl.col("CLASIFICACION_FINAL_COVID").is_in(["1", "2", "3"])
    & (pl.col("ENTIDAD_UM") == "09")
)
mask_faltantes = (
    pl.any_horizontal([pl.col(c) == "98" for c in comorbilidades])
    | (pl.col("NEUMONIA") == "99")
    | (pl.col("EMBARAZO") == "98")
)
mask_outliers_ok = (pl.col("EDAD") <= 110) & (pl.col("DIAS_SINTOMAS") <= 30)
 
lf_base = lf_raw.filter(filtro_confirmados_cdmx)                  # Paso 3.1 + 3.2
lf_sin_faltantes = lf_base.filter(~mask_faltantes)                # Paso 3.3
lf_sin_outliers = lf_sin_faltantes.filter(mask_outliers_ok)        # Paso 3.4
 
# =================================================================
# PIPELINE 1: TRANSFORMACION
# =================================================================
lf_transform = lf_sin_outliers.with_columns(
    [
        pl.when(pl.col(c) == "1").then(1).otherwise(0).cast(pl.Int8).alias(c)
        for c in comorbilidades + ["NEUMONIA"]
    ] + [
        pl.when(pl.col("EMBARAZO") == "1").then(1).otherwise(0).cast(pl.Int8).alias("EMBARAZO"),
        pl.when(pl.col("SEXO") == "2").then(1).otherwise(0).cast(pl.Int8).alias("es_hombre"),
        pl.when(pl.col("TIPO_PACIENTE") == "2").then(1).otherwise(0).cast(pl.Int8).alias("hospitalizado"),
        (pl.col("ENTIDAD_RES") == "09").cast(pl.Int8).alias("reside_cdmx"),
    ]
).select([
    "EDAD", "es_hombre", "EMBARAZO", "DIAS_SINTOMAS", "reside_cdmx",
    "NEUMONIA",  # condicional: solo Modelo A
    *comorbilidades,
    "hospitalizado",  # y
    "MUNICIPIO_RES",  # descriptiva, no predictora
])
 
df_limpio = lf_transform.collect(engine="streaming")
 
print(df_limpio.shape)
print(df_limpio["hospitalizado"].value_counts(sort=True))
print(df_limpio.null_count())
print(df_limpio.head())


(2139591, 18)
shape: (2, 2)
┌───────────────┬─────────┐
│ hospitalizado ┆ count   │
│ ---           ┆ ---     │
│ i8            ┆ u32     │
╞═══════════════╪═════════╡
│ 0             ┆ 2004256 │
│ 1             ┆ 135335  │
└───────────────┴─────────┘
shape: (1, 18)
┌──────┬───────────┬──────────┬───────────────┬─────────────┬──────────┬──────────┬──────────────┬──────────┬──────┬──────┬────────────────┬───────────────┬──────────┬────────────┬──────────┬───────────────┬───────────────┐
│ EDAD ┆ es_hombre ┆ EMBARAZO ┆ DIAS_SINTOMAS ┆ reside_cdmx ┆ NEUMONIA ┆ DIABETES ┆ HIPERTENSION ┆ OBESIDAD ┆ EPOC ┆ ASMA ┆ CARDIOVASCULAR ┆ RENAL_CRONICA ┆ INMUSUPR ┆ TABAQUISMO ┆ OTRA_COM ┆ hospitalizado ┆ MUNICIPIO_RES │
│ ---  ┆ ---       ┆ ---      ┆ ---           ┆ ---         ┆ ---      ┆ ---      ┆ ---          ┆ ---      ┆ ---  ┆ ---  ┆ ---            ┆ ---           ┆ ---      ┆ ---        ┆ ---      ┆ ---           ┆ ---           │
│ u32  ┆ u32       ┆ u32      ┆ u32           ┆ u32         ┆

In [5]:

CORRER_AUDITORIA = True
 
if CORRER_AUDITORIA:
 
    def contar(lf: pl.LazyFrame) -> int:
        return lf.select(pl.len()).collect(engine="streaming").item()
 
    filas_totales = contar(lf_raw)
    filas_confirmados_cdmx = contar(lf_base)
    filas_sin_faltantes = contar(lf_sin_faltantes)
    filas_sin_outliers = contar(lf_sin_outliers)
 
    print(f"Confirmados + CDMX: {filas_confirmados_cdmx:,}")
    print(f"Eliminadas por faltantes reales: {filas_confirmados_cdmx - filas_sin_faltantes:,}")
    print(f"Eliminadas por outliers: {filas_sin_faltantes - filas_sin_outliers:,}")
 
    filas_unicas = lf_raw.unique().select(pl.len()).collect(engine="streaming").item()
    print(f"Filas totales: {filas_totales:,} | Filas únicas: {filas_unicas:,}")


Confirmados + CDMX: 2,165,870
Eliminadas por faltantes reales: 25,612
Eliminadas por outliers: 667
Filas totales: 7,840,211 | Filas únicas: 7,840,211


In [6]:
output_path = Path().cwd().parent / "data" / "polars" / "interim" / "hospitalizacion_limpio.parquet"
output_path.parent.mkdir(parents=True, exist_ok=True)
df_limpio.write_parquet(output_path)
print(f"Guardado: {output_path} | shape: {df_limpio.shape}")

Guardado: /home/netfoor/Documents/university/KDD/covid19/data/polars/interim/hospitalizacion_limpio.parquet | shape: (2139591, 18)


In [1]:
import polars as pl
from pathlib import Path

output_path = Path().cwd().parent / "data" / "polars" / "interim" / "hospitalizacion_limpio.parquet"

df_schema = pl.scan_parquet(output_path).collect_schema()
print(df_schema)
print(pl.scan_parquet(output_path).collect_schema().len(), "columnas en el archivo limpio")
print(pl.scan_parquet(output_path).select(pl.len()).collect().item(), "filas en el archivo limpio")

Schema({'EDAD': Int64, 'es_hombre': Int8, 'EMBARAZO': Int8, 'DIAS_SINTOMAS': Int64, 'reside_cdmx': Int8, 'NEUMONIA': Int8, 'DIABETES': Int8, 'HIPERTENSION': Int8, 'OBESIDAD': Int8, 'EPOC': Int8, 'ASMA': Int8, 'CARDIOVASCULAR': Int8, 'RENAL_CRONICA': Int8, 'INMUSUPR': Int8, 'TABAQUISMO': Int8, 'OTRA_COM': Int8, 'hospitalizado': Int8, 'MUNICIPIO_RES': String})
18 columnas en el archivo limpio
2139591 filas en el archivo limpio


In [2]:
import polars as pl
from pathlib import Path

path = Path().cwd().parent / "data" / "polars" / "interim" / "hospitalizacion_limpio.parquet"
df = pl.read_parquet(path)

comorbilidades_core = [
    "DIABETES", "EPOC", "ASMA", "INMUSUPR",
    "HIPERTENSION", "CARDIOVASCULAR", "OBESIDAD", "RENAL_CRONICA",
]

df_features = df.with_columns([
    (pl.col("EDAD") >= 60).cast(pl.Int8).alias("adultoMayor"),
    pl.sum_horizontal(comorbilidades_core).cast(pl.Int8).alias("totalComorbilidades"),
    ((pl.col("DIABETES") + pl.col("HIPERTENSION") + pl.col("OBESIDAD")) >= 2)
        .cast(pl.Int8).alias("riesgoMetabolico"),
])

nuevas = ["adultoMayor", "totalComorbilidades", "riesgoMetabolico"]
print(df_features.select(nuevas).describe())
print()

for c in nuevas:
    print(f"--- {c} vs hospitalizado ---")
    print(
        df_features.group_by(c)
        .agg(pl.col("hospitalizado").mean().round(4).alias("tasa_hospitalizacion"), pl.len())
        .sort(c)
    )
    print()

shape: (9, 4)
┌────────────┬─────────────┬─────────────────────┬──────────────────┐
│ statistic  ┆ adultoMayor ┆ totalComorbilidades ┆ riesgoMetabolico │
│ ---        ┆ ---         ┆ ---                 ┆ ---              │
│ str        ┆ f64         ┆ f64                 ┆ f64              │
╞════════════╪═════════════╪═════════════════════╪══════════════════╡
│ count      ┆ 2.139591e6  ┆ 2.139591e6          ┆ 2.139591e6       │
│ null_count ┆ 0.0         ┆ 0.0                 ┆ 0.0              │
│ mean       ┆ 0.142325    ┆ 0.309296            ┆ 0.055611         │
│ std        ┆ 0.349383    ┆ 0.664376            ┆ 0.229169         │
│ min        ┆ 0.0         ┆ 0.0                 ┆ 0.0              │
│ 25%        ┆ 0.0         ┆ 0.0                 ┆ 0.0              │
│ 50%        ┆ 0.0         ┆ 0.0                 ┆ 0.0              │
│ 75%        ┆ 0.0         ┆ 0.0                 ┆ 0.0              │
│ max        ┆ 1.0         ┆ 8.0                 ┆ 1.0              │
└─────

In [3]:
# --- ¿qué distingue a los pacientes con 6, 7 y 8 comorbilidades? ---
print(
    df_features.filter(pl.col("totalComorbilidades") >= 6)
    .group_by("totalComorbilidades")
    .agg(
        pl.col("EDAD").mean().round(1).alias("edad_media"),
        pl.col("EDAD").median().alias("edad_mediana"),
        pl.col("adultoMayor").mean().round(3).alias("pct_adultoMayor"),
        pl.col("NEUMONIA").mean().round(3).alias("pct_neumonia"),
        pl.col("hospitalizado").mean().round(4).alias("tasa_hosp"),
        pl.len(),
    )
    .sort("totalComorbilidades")
)

shape: (3, 7)
┌───────────────────┬────────────┬──────────────┬─────────────────┬──────────────┬───────────┬─────┐
│ totalComorbilidad ┆ edad_media ┆ edad_mediana ┆ pct_adultoMayor ┆ pct_neumonia ┆ tasa_hosp ┆ len │
│ es                ┆ ---        ┆ ---          ┆ ---             ┆ ---          ┆ ---       ┆ --- │
│ ---               ┆ f64        ┆ f64          ┆ f64             ┆ f64          ┆ f64       ┆ u32 │
│ i8                ┆            ┆              ┆                 ┆              ┆           ┆     │
╞═══════════════════╪════════════╪══════════════╪═════════════════╪══════════════╪═══════════╪═════╡
│ 6                 ┆ 63.0       ┆ 64.0         ┆ 0.632           ┆ 0.333        ┆ 0.5088    ┆ 171 │
│ 7                 ┆ 52.2       ┆ 56.0         ┆ 0.409           ┆ 0.318        ┆ 0.3864    ┆ 44  │
│ 8                 ┆ 37.7       ┆ 36.5         ┆ 0.095           ┆ 0.101        ┆ 0.0893    ┆ 168 │
└───────────────────┴────────────┴──────────────┴─────────────────┴──────────

In [4]:
# --- Selección final y guardado del parquet "processed" ---
predictoras_base = [
    "EDAD", "adultoMayor", "es_hombre", "EMBARAZO", "DIAS_SINTOMAS", "reside_cdmx",
    "DIABETES", "HIPERTENSION", "OBESIDAD", "EPOC", "ASMA",
    "CARDIOVASCULAR", "RENAL_CRONICA", "INMUSUPR", "TABAQUISMO", "OTRA_COM",
    "totalComorbilidades", "riesgoMetabolico",
]
columnas_finales = predictoras_base + ["NEUMONIA", "hospitalizado", "MUNICIPIO_RES"]

df_processed = df_features.select(columnas_finales)

output_path = Path().cwd().parent / "data" / "polars" / "processed" / "hospitalizacion_features.parquet"
output_path.parent.mkdir(parents=True, exist_ok=True)
df_processed.write_parquet(output_path)

print(f"Guardado: {output_path}")
print(df_processed.shape)
print(f"Modelo A (con NEUMONIA): {len(predictoras_base) + 1} predictoras")
print(f"Modelo B (sin NEUMONIA): {len(predictoras_base)} predictoras")

Guardado: /home/netfoor/Documents/university/KDD/covid19/data/polars/processed/hospitalizacion_features.parquet
(2139591, 21)
Modelo A (con NEUMONIA): 19 predictoras
Modelo B (sin NEUMONIA): 18 predictoras
